# 📘 04_lakehouse_monitoring_functions.ipynb

In this notebook, we create **custom SQL functions (UDFs)** that act as reusable building blocks for Databricks Lakehouse Monitoring.

Databricks Lakehouse Monitoring allows users to define **custom metrics** through SQL expressions or functions.  
While simple metrics like averages or null ratios can be expressed inline, enterprise-scale monitoring often involves repeating the same logic across hundreds of tables — such as:

- Ensuring `end_date >= start_date`
- Checking that `claim_status = 'CLOSED'` implies `closed_at IS NOT NULL`
- Detecting negative or out-of-range numeric values
- Verifying timestamps are not in the future

If each metric embeds this logic directly, maintenance quickly becomes painful and inconsistent.  
By defining SQL functions once and referencing them across metrics, you achieve **reusability**, **consistency**, and **auditability** — all governed under Unity Catalog.

### Why Use Functions

| Benefit | Description |
|----------|-------------|
| **Reusability** | Define logic once and reuse across hundreds of metrics. |
| **Consistency** | Every monitor references the same version of the rule logic. |
| **Auditability** | Functions are Unity Catalog–governed with lineage and versioning. |
| **Modularity** | Update logic centrally without touching metric definitions. |

Example — instead of embedding logic inline:
avg(dbdemos_steventan.monitoring_admin.rule_negative_amount_ratio_bit(amount))

we can reference the function directly in metric templates.

### What We’ll Implement

We’ll define a set of reusable SQL functions grouped by **data quality dimensions**, similar to frameworks like Great Expectations or Soda, but implemented natively in the Lakehouse.

| Dimension | Description | Example Function |
|------------|--------------|------------------|
| **Validity** | Ensure values follow business or logical rules | `rule_negative_amount_ratio_bit(amount)` |
| **Completeness** | Detect missing or empty data | `rule_missing_value_ratio_bit(val)` |
| **Consistency** | Validate relationships between columns | `rule_inconsistent_closed_claims_ratio_bit(status, closed_at)` |
| **Accuracy** | Flag values outside expected business ranges | `rule_premium_out_of_range_ratio_bit(amount)` |

Each rule has two variants:
- `*_ratio_bit` → returns `1.0` or `0.0` for violation ratio aggregation  
- `*_details` → returns a structured JSON object describing violations

### How It Fits in the Metadata Framework

These functions form the **foundation** of the metadata-driven Lakehouse Monitoring framework:

<pre>
┌────────────────────┐
│  SQL Functions     │  ← reusable rule logic (this notebook)
├────────────────────┤
│  Metric Templates  │  ← define metric expressions using these functions
├────────────────────┤
│  Metric Bindings   │  ← attach templates to specific tables
├────────────────────┤
│  Monitors Control  │  ← schedule, enable, and manage monitors
└────────────────────┘
</pre>

## 1️⃣ Widgets — Catalog and Schema Parameters

Define your working context (catalog, schemas, and asset directories).  
These widgets ensure this notebook can run across different workspaces without modification.

In [0]:
dbutils.widgets.text("catalog", "dbdemos_steventan", "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin", "Admin Schema")

catalog = dbutils.widgets.get("catalog")
admin_schema = dbutils.widgets.get("admin_schema")

In [0]:
# Uses existing widgets/vars: catalog, admin_schema
FUNC_SCHEMA = f"{catalog}.{admin_schema}"

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FUNC_SCHEMA}")

# Helper to fully-qualify function names
def F(name: str) -> str:
    return f"{FUNC_SCHEMA}.{name}"

## Validity rules

These catch values or combinations that are **invalid** with respect to business logic or formats:
- `end_before_start_*`: end date earlier than start date
- `future_timestamp_*`: timestamp lies in the future
- `negative_amount_*`: negative numeric amounts
- `invalid_date_string_*`: string cannot be parsed to a date (format aware)
- `unexpected_category_*`: value not in an expected set

In [0]:
stmts_validity = [

# 1) end_before_start
f"""CREATE OR REPLACE FUNCTION {F("rule_end_before_start_ratio_bit")}(
  startd DATE COMMENT 'Start date of the period/event',
  endd DATE COMMENT 'End date of the period/event'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies records where end date occurs before start date.
Returns 1.0 (invalid) if end < start, 0.0 (valid) otherwise.
Usage: avg(rule_end_before_start_ratio_bit(contract_start_date, contract_end_date))
Related: rule_end_before_start_details'
DETERMINISTIC
RETURN CASE WHEN endd < startd THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_end_before_start_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  startd DATE COMMENT 'Start date of the period/event',
  endd DATE COMMENT 'End date of the period/event'
)
RETURNS STRUCT<pk:STRING,start_date:DATE,end_date:DATE,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for records where end date is before start date.
Returns STRUCT for failed records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN endd < startd
            THEN named_struct('pk', pk, 'start_date', startd, 'end_date', endd, 'reason', 'end_before_start')
       END""",

# 2) future_timestamp
f"""CREATE OR REPLACE FUNCTION {F("rule_future_timestamp_ratio_bit")}(
  ts TIMESTAMP COMMENT 'Timestamp value to validate against current time'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies records with timestamps in the future.
Returns 1.0 (invalid) if timestamp > current_timestamp(), 0.0 (valid) otherwise.
Usage: avg(rule_future_timestamp_ratio_bit(created_at))
Related: rule_future_timestamp_details'
DETERMINISTIC
RETURN CASE WHEN ts > current_timestamp() THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_future_timestamp_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  ts TIMESTAMP COMMENT 'The future timestamp value'
)
RETURNS STRUCT<pk:STRING,ts:TIMESTAMP,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for records with future timestamps.
Returns STRUCT for failed records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN ts > current_timestamp()
            THEN named_struct('pk', pk, 'ts', ts, 'reason', 'future_timestamp')
       END""",

# 3) negative_amount
f"""CREATE OR REPLACE FUNCTION {F("rule_negative_amount_ratio_bit")}(
  amount DOUBLE COMMENT 'Numeric amount that should be positive'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies records with negative amounts where positive values are expected.
Returns 1.0 (invalid) if amount < 0, 0.0 (valid) otherwise.
Usage: avg(rule_negative_amount_ratio_bit(revenue_amount))
Related: rule_negative_amount_details'
DETERMINISTIC
RETURN CASE WHEN amount < 0 THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_negative_amount_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  amount DOUBLE COMMENT 'The negative amount value'
)
RETURNS STRUCT<pk:STRING,amount:DOUBLE,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for records with negative amounts.
Returns STRUCT for failed records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN amount < 0
            THEN named_struct('pk', pk, 'amount', amount, 'reason', 'negative_amount')
       END""",

# 4) invalid_date_string
f"""CREATE OR REPLACE FUNCTION {F("rule_invalid_date_string_ratio_bit")}(
  date_str STRING COMMENT 'String representation of a date',
  fmt STRING COMMENT 'Expected date format (e.g., yyyy-MM-dd, MM/dd/yyyy)'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies strings that cannot be parsed as valid dates with the given format.
Returns 1.0 (invalid) if cannot parse, 0.0 (valid) otherwise.
Usage: avg(rule_invalid_date_string_ratio_bit(birth_date_str, "yyyy-MM-dd"))
Related: rule_invalid_date_string_details'
DETERMINISTIC
RETURN CASE WHEN to_date(date_str, fmt) IS NULL THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_invalid_date_string_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  date_str STRING COMMENT 'The invalid date string',
  fmt STRING COMMENT 'Expected date format pattern'
)
RETURNS STRUCT<pk:STRING,date_str:STRING,fmt:STRING,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for malformed date strings.
Returns STRUCT for failed records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN to_date(date_str, fmt) IS NULL
            THEN named_struct('pk', pk, 'date_str', date_str, 'fmt', fmt, 'reason', 'invalid_date_string')
       END""",

# 5) unexpected_category
f"""CREATE OR REPLACE FUNCTION {F("rule_unexpected_category_ratio_bit")}(
  val STRING COMMENT 'Category/status value to validate',
  csv_expected STRING COMMENT 'Comma-separated list of valid values'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies records with category values not in the expected set.
Returns 1.0 (invalid) if not in expected set, 0.0 (valid) otherwise.
Usage: avg(rule_unexpected_category_ratio_bit(status, "active,pending,closed,cancelled"))
Related: rule_unexpected_category_details'
DETERMINISTIC
RETURN CASE WHEN NOT array_contains(split(csv_expected, ','), val) THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_unexpected_category_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  val STRING COMMENT 'The unexpected category value',
  csv_expected STRING COMMENT 'Comma-separated list of valid values'
)
RETURNS STRUCT<pk:STRING,val:STRING,expected_set:STRING,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for unexpected category values.
Returns STRUCT for failed records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN NOT array_contains(split(csv_expected, ','), val)
            THEN named_struct('pk', pk, 'val', val, 'expected_set', csv_expected, 'reason', 'unexpected_category')
       END""",
]

## Completeness rules

These detect **missingness** (null/empty) in mandatory fields.

In [0]:
stmts_completeness = [

# 6) missing_value (NULL or empty string)
f"""CREATE OR REPLACE FUNCTION {F("rule_missing_value_ratio_bit")}(
  val STRING COMMENT 'Column value to check for completeness'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies records with NULL or empty/whitespace-only string values.
Returns 1.0 (incomplete) if NULL or empty string, 0.0 (complete) otherwise.
Usage: avg(rule_missing_value_ratio_bit(customer_name))
Related: rule_missing_value_details'
DETERMINISTIC
RETURN CASE WHEN val IS NULL OR trim(val) = '' THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_missing_value_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  colname STRING COMMENT 'Name of the column being checked',
  val STRING COMMENT 'The missing/empty value'
)
RETURNS STRUCT<pk:STRING,colname:STRING,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for records with missing or empty values.
Returns STRUCT for incomplete records, NULL for complete records.'
DETERMINISTIC
RETURN CASE WHEN val IS NULL OR trim(val) = ''
            THEN named_struct('pk', pk, 'colname', colname, 'reason', 'missing_value')
       END""",
]

## Consistency rules

These enforce **logical consistency** across related fields.

In [0]:
stmts_consistency = [

# 7) closed claims must have closed_at
f"""CREATE OR REPLACE FUNCTION {F("rule_inconsistent_closed_claims_ratio_bit")}(
  claim_status STRING COMMENT 'Current status of the claim',
  closed_at TIMESTAMP COMMENT 'Timestamp when the claim was closed'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies claims marked as CLOSED but missing a closed_at timestamp.
Returns 1.0 (inconsistent) if status=CLOSED and closed_at is NULL, 0.0 (consistent) otherwise.
Usage: avg(rule_inconsistent_closed_claims_ratio_bit(status, closed_at))
Related: rule_inconsistent_closed_claims_details'
DETERMINISTIC
RETURN CASE WHEN claim_status = 'CLOSED' AND closed_at IS NULL THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_inconsistent_closed_claims_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  claim_status STRING COMMENT 'The status showing CLOSED',
  closed_at TIMESTAMP COMMENT 'The missing closure timestamp'
)
RETURNS STRUCT<pk:STRING,claim_status:STRING,closed_at:TIMESTAMP,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for closed claims without closure dates.
Returns STRUCT for inconsistent records, NULL for consistent records.'
DETERMINISTIC
RETURN CASE WHEN claim_status = 'CLOSED' AND closed_at IS NULL
            THEN named_struct('pk', pk, 'claim_status', claim_status, 'closed_at', closed_at, 'reason', 'inconsistent_closed_claims')
       END""",
]

## Accuracy rules

These flag values that are **outside expected business ranges**.

In [0]:
stmts_accuracy = [

# 8) premium amount out of expected business range
f"""CREATE OR REPLACE FUNCTION {F("rule_premium_out_of_range_ratio_bit")}(
  amount DOUBLE COMMENT 'Premium amount to validate against business rules'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies premium amounts outside expected business range ($100 - $100,000).
Returns 1.0 (inaccurate) if amount < $100 or > $100,000, 0.0 (accurate) otherwise.
Usage: avg(rule_premium_out_of_range_ratio_bit(premium_amount))
Related: rule_premium_out_of_range_details'
DETERMINISTIC
RETURN CASE WHEN amount < 100 OR amount > 100000 THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_premium_out_of_range_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  amount DOUBLE COMMENT 'The out-of-range premium amount'
)
RETURNS STRUCT<pk:STRING,amount:DOUBLE,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for premiums outside business range.
Returns STRUCT for out-of-range records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN amount < 100 OR amount > 100000
            THEN named_struct('pk', pk, 'amount', amount, 'reason', 'premium_out_of_range')
       END""",

# 9) age out of reasonable range
f"""CREATE OR REPLACE FUNCTION {F("rule_age_out_of_range_ratio_bit")}(
  age INT COMMENT 'Age value to validate',
  min_age INT COMMENT 'Minimum valid age',
  max_age INT COMMENT 'Maximum valid age'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies age values outside reasonable human age range.
Returns 1.0 (inaccurate) if age < min_age or > max_age, 0.0 (accurate) otherwise.
Usage: avg(rule_age_out_of_range_ratio_bit(customer_age, 18, 120))
Related: rule_age_out_of_range_details'
DETERMINISTIC
RETURN CASE WHEN age < min_age OR age > max_age THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_age_out_of_range_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  age INT COMMENT 'The out-of-range age value',
  min_age INT COMMENT 'Minimum valid age used',
  max_age INT COMMENT 'Maximum valid age used'
)
RETURNS STRUCT<pk:STRING,age:INT,min_expected:INT,max_expected:INT,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for out-of-range ages.
Returns STRUCT for out-of-range records, NULL for valid records.'
DETERMINISTIC
RETURN CASE WHEN age < min_age OR age > max_age
            THEN named_struct('pk', pk, 'age', age, 'min_expected', min_age, 'max_expected', max_age, 'reason', 'age_out_of_range')
       END""",

# 10) value out of percentile range (dynamic range based on data distribution)
f"""CREATE OR REPLACE FUNCTION {F("rule_outlier_ratio_bit")}(
  val DOUBLE COMMENT 'Numeric value to check for outliers',
  lower_bound DOUBLE COMMENT 'Lower bound (e.g., P5 or mean - 3*std)',
  upper_bound DOUBLE COMMENT 'Upper bound (e.g., P95 or mean + 3*std)'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Identifies values outside statistical bounds (outliers).
Returns 1.0 (outlier) if val < lower_bound or > upper_bound, 0.0 (normal) otherwise.
Usage: avg(rule_outlier_ratio_bit(claim_amount, 500, 50000))
Related: rule_outlier_details'
DETERMINISTIC
RETURN CASE WHEN val < lower_bound OR val > upper_bound THEN 1.0 ELSE 0.0 END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_outlier_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  val DOUBLE COMMENT 'The outlier value',
  lower_bound DOUBLE COMMENT 'Lower bound used',
  upper_bound DOUBLE COMMENT 'Upper bound used'
)
RETURNS STRUCT<pk:STRING,value:DOUBLE,lower_bound:DOUBLE,upper_bound:DOUBLE,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for statistical outliers.
Returns STRUCT for outlier records, NULL for normal values.'
DETERMINISTIC
RETURN CASE WHEN val < lower_bound OR val > upper_bound
            THEN named_struct('pk', pk, 'value', val, 'lower_bound', lower_bound, 'upper_bound', upper_bound, 'reason', 'statistical_outlier')
       END""",

# 11) percentage value out of valid range (0-100 or 0-1)
f"""CREATE OR REPLACE FUNCTION {F("rule_invalid_percentage_ratio_bit")}(
  pct DOUBLE COMMENT 'Percentage value to validate',
  is_decimal BOOLEAN COMMENT 'TRUE if percentage is 0-1 decimal, FALSE if 0-100'
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Validates percentage values are within valid range (0-100 or 0-1).
Returns 1.0 (invalid) if outside valid range, 0.0 (valid) otherwise.
Usage: avg(rule_invalid_percentage_ratio_bit(completion_rate, TRUE))
Related: rule_invalid_percentage_details'
DETERMINISTIC
RETURN CASE 
  WHEN is_decimal AND (pct < 0 OR pct > 1) THEN 1.0
  WHEN NOT is_decimal AND (pct < 0 OR pct > 100) THEN 1.0
  ELSE 0.0 
END""",

f"""CREATE OR REPLACE FUNCTION {F("rule_invalid_percentage_details")}(
  pk STRING COMMENT 'Primary key or unique identifier',
  pct DOUBLE COMMENT 'The invalid percentage value',
  is_decimal BOOLEAN COMMENT 'TRUE if 0-1 range, FALSE if 0-100'
)
RETURNS STRUCT<pk:STRING,percentage:DOUBLE,expected_range:STRING,reason:STRING>
LANGUAGE SQL
COMMENT 'Returns detailed information for invalid percentages.
Returns STRUCT for invalid records, NULL for valid records.'
DETERMINISTIC
RETURN CASE 
  WHEN is_decimal AND (pct < 0 OR pct > 1) 
    THEN named_struct('pk', pk, 'percentage', pct, 'expected_range', '0-1', 'reason', 'percentage_out_of_range')
  WHEN NOT is_decimal AND (pct < 0 OR pct > 100) 
    THEN named_struct('pk', pk, 'percentage', pct, 'expected_range', '0-100', 'reason', 'percentage_out_of_range')
END""",
]

## Execute — Create/Replace all functions

Re-running this cell is **idempotent** (safe). It recreates the UDFs in `catalog.admin_schema`.

In [0]:
for s in (stmts_validity + stmts_completeness + stmts_consistency + stmts_accuracy):
    spark.sql(s)

print(f"✅ Created/updated custom metric functions in {FUNC_SCHEMA}")

### Next Step

After running this notebook:
1. You’ll have a library of SQL functions under `{catalog}.{admin_schema}`.
2. These will be referenced by **metric templates** in the next notebook (v0.2).
3. This approach enables modular, metadata-driven monitoring at scale — with clear separation of **rule logic**, **metric definition**, and **monitor orchestration**.